# PRNU-v2 reference-free binary usefulness gate

This notebook extracts a single-image PRNU-v2 runtime vector, runs the fail-closed 512px coverage audit, trains the PRNU-only diagnostic and RINE+PRNU candidate across seeds 42/43/44, and applies the locked retention gate. It never reads `final_test` and never compares an input with a known-device fingerprint.


In [1]:
import subprocess
from pathlib import Path

PROJECT = Path('/content/cya-techjam26')
REPO_URL = 'https://github.com/maxi-cmyk/cya-techjam26.git'
if PROJECT.is_dir():
    result = subprocess.run(['git', 'pull', '--ff-only'], cwd=PROJECT, capture_output=True, text=True)
else:
    result = subprocess.run(['git', 'clone', REPO_URL, str(PROJECT)], capture_output=True, text=True)
print(result.stdout, result.stderr)
assert result.returncode == 0, 'repository clone/pull failed'
required_prnu_files = (
    PROJECT / 'scripts/extract_prnu_runtime_v2.py',
    PROJECT / 'scripts/train_prnu_runtime_v2.py',
    PROJECT / 'src/cya_detector/features/prnu_runtime_v2.py',
)
missing_prnu_files = [str(path.relative_to(PROJECT)) for path in required_prnu_files if not path.is_file()]
assert not missing_prnu_files, (
    'The Colab checkout does not contain the Notebook 08 implementation. '
    'Commit and push the local PRNU-v2 changes before running this notebook. '
    f'Missing: {missing_prnu_files}'
)


 Cloning into '/content/cya-techjam26'...



In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import json
import shutil

ROBUSTNESS = PROJECT / 'artifacts/robustness'
PRIOR = Path('/content/drive/MyDrive/cya-techjam26/artifacts')
DURABLE = PRIOR / 'robustness'
SEEDS = (42, 43, 44)

def run_make(*args):
    result = subprocess.run(['make', *args], cwd=PROJECT, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"make {' '.join(args)} failed with code {result.returncode}")
    return result

def run_script(*args):
    result = subprocess.run(list(args), cwd=PROJECT, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"{' '.join(args)} failed with code {result.returncode}")
    return result


In [4]:
run_make('install-colab')
import torch
assert torch.cuda.is_available(), (
    'Notebook 08 requires a GPU runtime for frozen RINE/CLIP extraction. '
    'In Colab select Runtime > Change runtime type > GPU, then rerun from the top.'
)
print('GPU:', torch.cuda.get_device_name(0))
run_make('smoke-bootstrap')
run_script('python', '-m', 'unittest', 'tests.test_prnu_runtime_v2', 'tests.test_robustness_training', '-v')


python -m pip install -r requirements-colab.txt
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 96.3 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 5.0.0.93
    Uninstalling opencv-python-headless-5.0.0.93:
      Successfully uninstalled opencv-python-headless-5.0.0.93
python -m pip install -e . --no-deps
Obtaining file:///content/cya-techjam26
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'do

CompletedProcess(args=['python', '-m', 'unittest', 'tests.test_prnu_runtime_v2', 'tests.test_robustness_training', '-v'], returncode=0, stdout='', stderr='test_prnu_only_diagnostic_uses_complete_locked_selection_bank (tests.test_prnu_runtime_v2.PrnuRuntimeV2Tests.test_prnu_only_diagnostic_uses_complete_locked_selection_bank) ... ok\ntest_readiness_is_balanced_and_extraction_never_reads_final_test (tests.test_prnu_runtime_v2.PrnuRuntimeV2Tests.test_readiness_is_balanced_and_extraction_never_reads_final_test) ... \nPRNU v2 runtime:   0%|          | 0/4 [00:00<?, ?image/s]\nPRNU v2 runtime:  75%|███████▌  | 3/4 [00:00<00:00, 28.94image/s]\nPRNU v2 runtime: 100%|██████████| 4/4 [00:00<00:00, 34.71image/s]\nok\ntest_runtime_vector_is_reference_free_and_small_images_fail_closed (tests.test_prnu_runtime_v2.PrnuRuntimeV2Tests.test_runtime_vector_is_reference_free_and_small_images_fail_closed) ... ok\ntest_bank_rejects_missing_cells_crossed_labels_and_final_test (tests.test_robustness_training.

In [5]:
import csv
from concurrent.futures import ThreadPoolExecutor, as_completed

fixed_manifest = PRIOR / 'task2/fixed_q96_manifest.csv'
source_manifest = PRIOR / 'task2/source_manifest_split.csv'
drive_images = Path('/content/drive/MyDrive/hackathon_data/raw/sid_set/images')
local_images = Path('/content/hackathon_data/raw/sid_set/images')
assert fixed_manifest.is_file() and source_manifest.is_file() and drive_images.is_dir()
with fixed_manifest.open(newline='') as stream:
    fixed_rows = list(csv.DictReader(stream))

def stage_selected_source(row):
    destination = local_images / Path(row['source_path']).name
    source = drive_images / destination.name
    assert source.is_file(), source
    destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.exists() or destination.stat().st_size != source.stat().st_size:
        temporary = destination.with_suffix(destination.suffix + '.part')
        shutil.copy2(source, temporary)
        temporary.replace(destination)
    return destination

assert len(fixed_rows) == 2000, f'Expected 2,000 fixed-q96 rows, found {len(fixed_rows)}'
fixed_basenames = [Path(row['source_path']).name for row in fixed_rows]
assert len(fixed_basenames) == len(set(fixed_basenames)), 'Selected source basenames are not unique'

with ThreadPoolExecutor(max_workers=8) as executor:
    futures = [executor.submit(stage_selected_source, row) for row in fixed_rows]
    for completed, future in enumerate(as_completed(futures), start=1):
        future.result()
        if completed % 250 == 0 or completed == len(futures):
            print(f'Staged {completed}/{len(futures)} selected raw sources')

local_source_manifest = PROJECT / 'artifacts/task2/source_manifest_split_local.csv'
local_source_manifest.parent.mkdir(parents=True, exist_ok=True)
with source_manifest.open(newline='') as source_stream:
    reader = csv.DictReader(source_stream)
    source_rows = list(reader)
    source_fields = reader.fieldnames
assert source_fields and 'source_path' in source_fields
for row in source_rows:
    row['source_path'] = str(local_images / Path(row['source_path']).name)
with local_source_manifest.open('w', newline='') as output_stream:
    writer = csv.DictWriter(output_stream, fieldnames=source_fields)
    writer.writeheader()
    writer.writerows(source_rows)

regenerated_manifest = PROJECT / 'artifacts/task2/fixed_q96_manifest_regenerated.csv'
run_script(
    'python', 'scripts/build_matched_clean.py',
    '--source-manifest', str(local_source_manifest),
    '--output-root', str(PROJECT / 'artifacts/task2/matched_candidates'),
    '--output-manifest', str(regenerated_manifest),
    '--report', str(PROJECT / 'artifacts/task2/fixed_q96_report_regenerated.json'),
    '--policy', 'fixed_q96', '--seed', '42', '--limit-per-label', '1000',
)
print('PASS: only the 2,000 selected Task 2 sources were staged and matched-clean views regenerated.')


Staged 250/2000 selected raw sources
Staged 500/2000 selected raw sources
Staged 750/2000 selected raw sources
Staged 1000/2000 selected raw sources
Staged 1250/2000 selected raw sources
Staged 1500/2000 selected raw sources
Staged 1750/2000 selected raw sources
Staged 2000/2000 selected raw sources
{
  "encoder_version": "Pillow-11.3.0",
  "image_count": 2000,
  "label_counts": {
    "ai_generated": 1000,
    "authentic": 1000
  },
  "limit_per_label": 1000,
  "metadata_policy": "strip_exif",
  "output_manifest": "/content/cya-techjam26/artifacts/task2/fixed_q96_manifest_regenerated.csv",
  "output_manifest_sha256": "61169374947286a0b25d855ea7039d28a49ed3f1eeb18f2d6ae7e9a501679782",
  "policy": "fixed_q96",
  "quality_counts": {
    "96": 2000
  },
  "resize_applied": false,
  "seed": 42,
  "source_manifest": "/content/cya-techjam26/artifacts/task2/source_manifest_split_local.csv",
  "source_manifest_sha256": "a3dc5e48330124daa5d0ca13a01075b010916f8c698924116dc4f2c9f4eec630",
  "subsa

In [4]:
from pathlib import Path
DURABLE = Path('/content/drive/MyDrive/cya-techjam26/artifacts/robustness')

In [5]:
import csv
import shutil
import subprocess
from pathlib import Path

# Re-resolve this cell's prerequisites after any harmless kernel-state reset.
PROJECT = Path('/content/cya-techjam26')
PRIOR = Path('/content/drive/MyDrive/cya-techjam26/artifacts')
ROBUSTNESS = PROJECT / 'artifacts/robustness'
DURABLE = PRIOR / 'robustness'
SEEDS = (42, 43, 44)
regenerated_manifest = PROJECT / 'artifacts/task2/fixed_q96_manifest_regenerated.csv'

def run_make(*args):
    result = subprocess.run(['make', *args], cwd=PROJECT, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"make {' '.join(args)} failed with code {result.returncode}")
    return result

assert PROJECT.is_dir(), f'Repository checkout not found: {PROJECT}'
assert DURABLE.is_dir(), (
    f'Notebook 07 artifacts not found: {DURABLE}. Mount Drive and verify '
    'MyDrive/cya-techjam26/artifacts/robustness exists.'
)
assert regenerated_manifest.is_file(), (
    f'Regenerated Task 2 manifest not found: {regenerated_manifest}. '
    'Run the preceding source-staging/regeneration cell first.'
)
# Rebuild the deterministic transform bank on Colab local storage. Copying its
# 19,460 individual images from mounted Drive can take well over an hour.
run_make('robustness-prepare', f'TASK2_SELECTED_MANIFEST={regenerated_manifest}')
parent_filenames = ('best_50_50.pt', 'best_50_50_predictions.csv')
for seed in SEEDS:
    durable_seed = DURABLE / 'train-controlled-rine' / f'seed_{seed}'
    local_seed = ROBUSTNESS / 'train-controlled-rine' / f'seed_{seed}'
    local_seed.mkdir(parents=True, exist_ok=True)
    for filename in parent_filenames:
        source = durable_seed / filename
        assert source.is_file(), f'Notebook 07 parent artifact not found: {source}'
        shutil.copy2(source, local_seed / filename)
combined_manifest = ROBUSTNESS / 'manifests/combined_manifest.csv'
assert combined_manifest.is_file()
for seed in SEEDS:
    assert (ROBUSTNESS / 'train-controlled-rine' / f'seed_{seed}' / 'best_50_50.pt').is_file()
with combined_manifest.open(newline='') as stream:
    combined_rows = list(csv.DictReader(stream))
assert combined_rows, 'Combined robustness manifest is empty'
assert all(row['split'] != 'final_test' for row in combined_rows), 'final_test leaked into Notebook 08'
allowed_image_roots = (PROJECT / 'artifacts/task2/matched_candidates', ROBUSTNESS / 'variants')
unexpected_paths = []
missing_paths = []
for row in combined_rows:
    image_path = Path(row['image_path'])
    if not any(image_path.is_relative_to(root) for root in allowed_image_roots):
        unexpected_paths.append(str(image_path))
    elif not image_path.is_file():
        missing_paths.append(str(image_path))
assert not unexpected_paths, f'Unexpected manifest image paths: {unexpected_paths[:5]}'
assert not missing_paths, f'Missing manifest images: {missing_paths[:5]} (total={len(missing_paths)})'
print(f'PASS: restored {len(combined_rows):,} manifest images and all three controlled-RINE parents.')


KeyboardInterrupt: 

In [6]:
run_make('robustness-prnu-v2-extract')
readiness = json.loads((ROBUSTNESS / 'features/prnu_v2_runtime_extraction_report.json').read_text())
assert readiness['ready_for_binary_ablation']
assert not readiness['reference_comparison_used']
assert not readiness['final_test_read']
print(readiness['groups'])


RuntimeError: make robustness-prnu-v2-extract failed with code 2

In [ ]:
for seed in SEEDS:
    run_make('robustness-prnu-v2-train', f'ROBUSTNESS_SEED={seed}')


In [ ]:
for seed in SEEDS:
    run_make('robustness-prnu-v2-fusion', f'ROBUSTNESS_SEED={seed}')


In [ ]:
run_make('robustness-prnu-v2-compare')
decision_path = ROBUSTNESS / 'reports/prnu_v2/retention_decision.json'
decision = json.loads(decision_path.read_text())
print(json.dumps(decision, indent=2, sort_keys=True))
assert decision['decision'] in {'retain', 'reject'}
assert not decision['final_test_read']


In [ ]:
def sync_tree(local_root: Path, remote_root: Path):
    remote_root.mkdir(parents=True, exist_ok=True)
    copied = 0
    for local_path in local_root.rglob('*'):
        relative = local_path.relative_to(local_root)
        remote_path = remote_root / relative
        if local_path.is_dir():
            remote_path.mkdir(parents=True, exist_ok=True)
        elif not remote_path.exists() or remote_path.stat().st_size != local_path.stat().st_size:
            remote_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(local_path, remote_path)
            copied += 1
    return copied

output_paths = (
    Path('features/prnu_v2_runtime_features.csv'),
    Path('features/prnu_v2_runtime_extraction_report.json'),
    Path('prnu_v2_runtime'),
    Path('rine_prnu_v2'),
    Path('reports/prnu_v2'),
)
copied = 0
for relative in output_paths:
    local_path = ROBUSTNESS / relative
    remote_path = DURABLE / relative
    assert local_path.exists(), f'Expected Notebook 08 output missing: {local_path}'
    if local_path.is_dir():
        copied += sync_tree(local_path, remote_path)
    elif not remote_path.exists() or remote_path.stat().st_size != local_path.stat().st_size:
        remote_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(local_path, remote_path)
        copied += 1
print(f'Synced {copied} Notebook 08 output files to {DURABLE}')
